# burn2scar quickstart: one tile, its mask, and the baseline model

This notebook downloads one tile of [burn2scar](https://doi.org/10.5281/zenodo.23986827) version 3.0 from
Hugging Face (about 3 MB), shows its image and mask, runs the released U-Net baseline on it, and compares the
prediction with the mask.

```bash
pip install huggingface_hub rasterio pandas matplotlib segmentation-models-pytorch safetensors
```

If you downloaded and unzipped the Zenodo release, set `LOCAL_ROOT` to that folder and the data are read from
there instead.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from huggingface_hub import hf_hub_download
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

DATASET = "neet1797/burn2scar"           # the data set on Hugging Face
MODEL = "neet1797/burn2scar-unet-r34"    # the released baseline model
LOCAL_ROOT = os.environ.get("BURN2SCAR_ROOT", "")  # an unzipped Zenodo release, if you have one


def fetch(path):
    """A file of the release: from LOCAL_ROOT if set, else downloaded once into the Hugging Face cache."""
    if LOCAL_ROOT:
        return Path(LOCAL_ROOT) / path
    return Path(hf_hub_download(DATASET, path, repo_type="dataset"))

## The index

One row per tile: paths, dates, place, class shares and the splits.

In [ ]:
index = pd.read_csv(fetch("index.csv"))
print(f"{len(index):,} tiles from {index['fire_id'].nunique():,} fires")
index.groupby(["country", "window"]).size().unstack()

## One test tile

A random tile from the test set of the main split, from a scene 5 to 60 days after its fire, with at least 10 %
fresh burn. Change `random_state` for another one.

In [ ]:
candidates = index[(index["split"] == "test") & (index["window"] == "fresh") & (index["frac_fresh"] >= 0.10)]
tile = candidates.sample(1, random_state=0).iloc[0]
tile[["tile_id", "country", "fire_date", "scene_time", "age_days", "area_ha", "frac_fresh", "frac_old"]]

In [ ]:
BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B10", "B11", "B12"]
CLASSES = {0: ("clear", "#f1efe8"), 2: ("fresh_burn", "#7f2704"), 3: ("old_burn", "#d9480f"), 4: ("cloud", "#b4c0ce"),
           5: ("cloud shadow", "#25272d"), 6: ("water", "#2a78d6"), 7: ("older_burn", "#f5a04a"), 255: ("no data", "#000000")}
values = sorted(CLASSES)
COLOURS = ListedColormap([CLASSES[v][1] for v in values])
NORM = BoundaryNorm([v - 0.5 for v in values] + [values[-1] + 0.5], COLOURS.N)


def composite(image, recipe):
    """An RGB picture from three bands, each divided by a brightness scale."""
    reflectance = image.astype(np.float32) / 10000
    return np.clip(np.stack([reflectance[BANDS.index(b)] / s for b, s in recipe], axis=-1), 0, 1) ** 0.9


with rasterio.open(fetch(tile["image_path"])) as src:
    image = src.read()          # 13 bands, top-of-atmosphere reflectance x 10,000, 0 = no data
with rasterio.open(fetch(tile["mask_path"])) as src:
    mask = src.read(1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4.3))
axes[0].imshow(composite(image, [("B04", .28), ("B03", .28), ("B02", .32)]))
axes[0].set_title("True colour")
axes[1].imshow(composite(image, [("B12", .40), ("B8A", .50), ("B04", .28)]))
axes[1].set_title("False colour (B12, B8A, B04)")
axes[2].imshow(mask, cmap=COLOURS, norm=NORM, interpolation="nearest")
axes[2].set_title("Mask")
present = [v for v in values if (mask == v).any()]
fig.legend(handles=[Patch(color=CLASSES[v][1], label=CLASSES[v][0]) for v in present], loc="lower center", ncol=len(present))
for ax in axes:
    ax.set_axis_off()
fig.suptitle(f"{tile['tile_id']}: {tile['country']}, {tile['age_days']:.0f} days after the fire")
plt.show()

## The baseline model

`predict.py` comes from the model repo. It reads any 13-band Level-1C image in 256 x 256 windows and returns
burn2scar mask values. The model never predicts older_burn, which was left out of training.

In [ ]:
sys.path.insert(0, str(Path(hf_hub_download(MODEL, "predict.py")).parent))
from predict import load_model, predict_array

net, config = load_model(MODEL)
predicted = predict_array(net, config, image)

fig, axes = plt.subplots(1, 2, figsize=(8, 4.3))
for ax, data, title in zip(axes, (mask, predicted), ("Mask", "Prediction")):
    ax.imshow(data, cmap=COLOURS, norm=NORM, interpolation="nearest")
    ax.set_title(title)
    ax.set_axis_off()
plt.show()

## How well it did on this tile

Scored as in the paper: fresh burn against clear, old burn and water, leaving cloud, shadow, older burn and no
data out. One tile says little; the paper reports the whole test set.

In [ ]:
scored = np.isin(mask, [0, 2, 3, 6])
truth, guess = mask[scored] == 2, predicted[scored] == 2
hits = int((truth & guess).sum())
precision = hits / max(int(guess.sum()), 1)
recall = hits / max(int(truth.sum()), 1)
f1 = 2 * hits / max(int(truth.sum() + guess.sum()), 1)
print(f"Fresh burn on this tile: precision {precision:.1%}, recall {recall:.1%}, F1 {f1:.1%}")

## Next

- Run the model on your own Sentinel-2 Level-1C images: `python predict.py --model neet1797/burn2scar-unet-r34 --image your_s2.tif --out your_pred.tif`. Bands in the order B01 to B12 with B8A after B08, reflectance x 10,000, 10 m grid.
- Train your own models with the splits in `index.csv`: `split` for the main results, `split_time` and `split_loco_*` for years and countries held out.
- Please cite the data set: Thind, P. K., Cadau, E., Del Prete, R., Ricolfi, S., Meoni, G., and Taramelli, A.: burn2scar: a Sentinel-2 burn-scar segmentation data set, version 3.0, Zenodo [data set], https://doi.org/10.5281/zenodo.23986827, 2026.